# Bài 9: Mạng nơ-ron nhiều lớp (MLP) – Thực hành trên Google Colab


In [ ]:
import os, shutil, zipfile
if not os.path.isdir("data"):
    try:
        from google.colab import files
        print("Hay chon tep Lab09_data.zip (hoac cac tep .csv cua bai)")
        up = files.upload()
    except ImportError:
        raise FileNotFoundError("Khong thay thu muc data/, hay dat no canh notebook")
    os.makedirs("data", exist_ok=True)
    for ten in up:
        if ten.endswith(".zip"):
            zipfile.ZipFile(ten).extractall(".")
        elif ten.endswith(".csv"):
            shutil.move(ten, os.path.join("data", ten))
print(sorted(os.listdir("data")))

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# ConvergenceWarning chi la loi nhac, an di cho de doc ket qua
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Tự tính lan truyền tiến


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

# Phan 1: mot no-ron voi ham sigmoid
w = np.array([0.8, -0.5, 0.3]); b = -0.2
x = np.array([2.0, 1.0, 3.0])
z = w @ x + b
print("Mot no-ron:")
print(f"  z = {w[0]}*{x[0]} + ({w[1]})*{x[1]} + {w[2]}*{x[2]} + ({b}) = {z:.2f}")
print(f"  a = sigmoid(z) = {sigmoid(z):.4f}")
print()

# Phan 2: mang giai XOR voi trong so tu dat
W1 = np.array([[1.0, 1.0],
               [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
W2 = np.array([1.0, -2.0]); b2 = 0.0

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
print("Mang giai XOR:")
print(" x1 x2  h1 h2  y")
for xx in X_xor:
    h = relu(xx @ W1 + b1)
    y_ = h @ W2 + b2
    print(f"  {xx[0]:.0f}  {xx[1]:.0f}   {h[0]:.0f}  {h[1]:.0f}  {y_:.0f}")
print()
H = relu(X_xor @ W1 + b1)
print("Tinh ca bang mot lan:", H @ W2 + b2)

## 2. Một nơ-ron và một lớp ẩn trên XOR


In [ ]:
xor = pd.read_csv("data/xor.csv")
X = xor[["x1", "x2"]]; y = xor["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)

lr = LogisticRegression().fit(X_tr, y_tr)
print(f"Hoi quy logistic: {accuracy_score(y_te, lr.predict(X_te)):.4f}")
print()
print("So no-ron an  so tham so  tap hoc  tap kiem tra")
for so in [1, 2, 4, 8]:
    mlp = MLPClassifier(hidden_layer_sizes=(so,), solver="lbfgs", max_iter=5000, random_state=0).fit(X_tr, y_tr)
    so_ts = sum(w_.size for w_ in mlp.coefs_) + sum(b_.size for b_ in mlp.intercepts_)
    print(f"{so:11d} {so_ts:11d} {accuracy_score(y_tr, mlp.predict(X_tr)):9.4f} {accuracy_score(y_te, mlp.predict(X_te)):13.4f}")

In [ ]:
def ve_ranh_gioi(mh, ax, Xv, yv, ten):
    xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 250), np.linspace(-1.1, 1.1, 250))
    zz = mh.predict(pd.DataFrame({"x1": xx.ravel(), "x2": yy.ravel()})).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#f4c7c7", "#c9d4e6"], alpha=0.8)
    ax.scatter(Xv["x1"], Xv["x2"], c=yv, cmap="coolwarm_r", s=10); ax.set_title(ten)

mlp8 = MLPClassifier(hidden_layer_sizes=(8,), solver="lbfgs", max_iter=5000, random_state=0).fit(X_tr, y_tr)
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
ve_ranh_gioi(lr, axs[0], X_tr, y_tr, "Hoi quy logistic, mot no-ron")
ve_ranh_gioi(mlp8, axs[1], X_tr, y_tr, "MLP, mot lop an 8 no-ron")
plt.show()

## 3. Độ lớn mạng và điều chuẩn


In [ ]:
moon = pd.read_csv("data/hai_vang_trang.csv")
Xm = moon[["x1", "x2"]]; ym = moon["nhan"]
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.25, random_state=17, stratify=ym)

def mang(lop_an, alpha, **kw):
    return make_pipeline(StandardScaler(),
        MLPClassifier(hidden_layer_sizes=lop_an, alpha=alpha, solver="lbfgs",
                      max_iter=20000, random_state=0, **kw))

print("Ba mang                 tap hoc  tap kiem tra")
for ten, lop_an, alpha in [("(2,), alpha 1e-4", (2,), 1e-4),
                           ("(100, 100), 1e-5", (100, 100), 1e-5),
                           ("(100, 100), 1.0", (100, 100), 1.0)]:
    m = mang(lop_an, alpha).fit(Xm_tr, ym_tr)
    print(f"  {ten:18s} {accuracy_score(ym_tr, m.predict(Xm_tr)):.4f}   {accuracy_score(ym_te, m.predict(Xm_te)):.4f}")
print()

print("alpha   kiem dinh cheo")
diem = {}
for alpha in [1e-5, 1e-3, 0.1, 1.0, 3.0, 10.0]:
    diem[alpha] = cross_val_score(mang((100, 100), alpha), Xm_tr, ym_tr, cv=5).mean()
    print(f"  {alpha:<8} {diem[alpha]:.4f}")
a_tot = max(diem, key=diem.get)
m = mang((100, 100), a_tot).fit(Xm_tr, ym_tr)
print(f"alpha tot nhat = {a_tot}, tap kiem tra {accuracy_score(ym_te, m.predict(Xm_te)):.4f}")

In [ ]:
sv = pd.read_csv("data/sinh_vien_mo_rong.csv")
Xs_ = sv[["gio_on", "diem_giua_ky", "chi_tieu"]]; ys_ = sv["qua_mon"]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs_, ys_, test_size=0.25, random_state=17, stratify=ys_)

tho = MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000, random_state=0).fit(Xs_tr, ys_tr)
chuan = make_pipeline(StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(20,), max_iter=3000, random_state=0)).fit(Xs_tr, ys_tr)

print(f"Doan bua ca lop qua : {ys_te.mean():.4f}")
print(f"Khong chuan hoa     : {accuracy_score(ys_te, tho.predict(Xs_te)):.4f}")
print(f"Co chuan hoa        : {accuracy_score(ys_te, chuan.predict(Xs_te)):.4f}")
print()
du = tho.predict(Xs_te)
print("Khong chuan hoa, so ban doan qua:", du.sum(), "tren", len(du))

## 5. Nhận dạng chữ số viết tay


In [ ]:
from sklearn.datasets import load_digits
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=17, stratify=yd)

mang_d = make_pipeline(StandardScaler(),
         MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=0)).fit(Xd_tr, yd_tr)
mlp_d = mang_d[-1]

print("Kich thuoc cac ma tran trong so:", [w_.shape for w_ in mlp_d.coefs_])
so_ts = sum(w_.size for w_ in mlp_d.coefs_) + sum(b_.size for b_ in mlp_d.intercepts_)
print("Tong so tham so:", so_ts)
print("So vong hoc:", mlp_d.n_iter_)
mm = mlp_d.loss_curve_
print(f"Mat mat vong 1: {mm[0]:.4f}, vong 10: {mm[9]:.4f}, vong cuoi: {mm[-1]:.4f}")
print()

du = mang_d.predict(Xd_te)
print(f"Accuracy tren tap kiem tra: {accuracy_score(yd_te, du):.4f}")
sai = np.flatnonzero(du != yd_te)
print("So anh doan sai:", len(sai), "tren", len(yd_te))
for i in sai:
    print(f"  anh {i:3d}: that la {yd_te[i]}, doan la {du[i]}")
print()

dung_som = make_pipeline(StandardScaler(),
           MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, early_stopping=True, random_state=0)).fit(Xd_tr, yd_tr)
print("Dung som sau", dung_som[-1].n_iter_, "vong, accuracy", f"{accuracy_score(yd_te, dung_som.predict(Xd_te)):.4f}")
print()

khac = {"Logistic co chuan hoa": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "KNN, k = 5": KNeighborsClassifier(),
        "SVM kernel RBF": SVC()}
for ten, mh in khac.items():
    print(f"{ten:22s} {accuracy_score(yd_te, mh.fit(Xd_tr, yd_tr).predict(Xd_te)):.4f}")

In [ ]:
plt.figure(figsize=(5, 3.2))
plt.plot(mlp_d.loss_curve_); plt.xlabel("Vong hoc"); plt.ylabel("Mat mat"); plt.title("Mat mat tren tap hoc giam dan"); plt.show()

---
# Bài tập


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(8,), solver="lbfgs", max_iter=5000, random_state=0).fit(X_tr, y_tr)
W1_, W2_ = mlp.coefs_
b1_, b2_ = mlp.intercepts_

nam_diem = X_te.iloc[:5].to_numpy()
h = relu(nam_diem @ W1_ + b1_)                 # lop an: ReLU
p_tay = sigmoid(h @ W2_ + b2_).ravel()         # lop ra: sigmoid
p_lib = mlp.predict_proba(X_te.iloc[:5])[:, 1]
print("Tu tinh    :", np.round(p_tay, 8))
print("predict_proba:", np.round(p_lib, 8))
print("Khop nhau:", np.allclose(p_tay, p_lib))

## Bài tập 2: Một mạng XOR khác


In [ ]:
for xx in X_xor:
    h1 = max(0, xx[0] - xx[1]); h2 = max(0, xx[1] - xx[0])
    print(f"x = ({xx[0]:.0f}, {xx[1]:.0f}) -> h1 = {h1:.0f}, h2 = {h2:.0f}, y = {h1 + h2:.0f}")

## Bài tập 3: So các hàm kích hoạt


In [ ]:
for act in ["logistic", "tanh", "relu"]:
    m = mang((100, 100), 1.0, activation=act).fit(Xm_tr, ym_tr)
    print(f"{act:9s} tap hoc {accuracy_score(ym_tr, m.predict(Xm_tr)):.4f}  tap kiem tra {accuracy_score(ym_te, m.predict(Xm_te)):.4f}")

## Bài tập 4: Tốc độ học


In [ ]:
plt.figure(figsize=(6, 4))
for lr_ in [0.0001, 0.001, 0.01, 0.1]:
    mh = make_pipeline(StandardScaler(),
         MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=lr_, max_iter=1000, random_state=0)).fit(Xd_tr, yd_tr)
    m_ = mh[-1]
    print(f"lr = {lr_:<7} n_iter {m_.n_iter_:4d}  mat mat cuoi {m_.loss_:.4f}  accuracy {accuracy_score(yd_te, mh.predict(Xd_te)):.4f}")
    plt.plot(m_.loss_curve_, label=f"lr = {lr_}")
plt.xlabel("Vong hoc"); plt.ylabel("Mat mat"); plt.yscale("log"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("toc_do_hoc.png", dpi=150, bbox_inches="tight"); plt.show()

## Bài tập 5: Dò lưới cho mạng nơ-ron


In [ ]:
pipe = make_pipeline(StandardScaler(), MLPClassifier(max_iter=500, random_state=0))
luoi = {"mlpclassifier__hidden_layer_sizes": [(10,), (50,), (100,), (50, 50)],
        "mlpclassifier__alpha": [0.0001, 0.01, 1]}
tim = GridSearchCV(pipe, luoi, cv=5, n_jobs=-1).fit(Xd_tr, yd_tr)
print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")
print(f"Accuracy tap kiem tra: {tim.score(Xd_te, yd_te):.4f}")
print(f"SVM kernel RBF       : {accuracy_score(yd_te, SVC().fit(Xd_tr, yd_tr).predict(Xd_te)):.4f}")

## Bài tập 6: Mạng nơ-ron cho hồi quy


In [ ]:
nha = pd.read_csv("data/gia_nha.csv")
Xn = nha.drop(columns="gia"); yn = nha["gia"]
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.25, random_state=17)

tt = LinearRegression().fit(Xn_tr, yn_tr)
mlp_r = make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(20,), max_iter=5000, random_state=0)).fit(Xn_tr, yn_tr)
print(f"MAE hoi quy tuyen tinh: {mean_absolute_error(yn_te, tt.predict(Xn_te)):.4f}")
print(f"MAE MLPRegressor      : {mean_absolute_error(yn_te, mlp_r.predict(Xn_te)):.4f}")

can = pd.DataFrame({"dien_tich": [250], "so_phong": [Xn["so_phong"].median()], "tuoi_nha": [Xn["tuoi_nha"].median()]})
print("Dien tich lon nhat trong du lieu:", Xn["dien_tich"].max())
print(f"Du doan can 250 m2 - tuyen tinh: {tt.predict(can)[0]:.2f} | MLP: {mlp_r.predict(can)[0]:.2f}")